In [26]:
!uv pip install langchain-ollama

Checked 1 package in 69ms


In [27]:
!uv pip install -qU "langchain-chroma"

In [28]:
#import required library

import os
import sys
from pathlib import Path

#langchain documnent loader
from langchain_community.document_loaders import PyPDFLoader

#lanchain text splitter
from langchain_text_splitters import RecursiveCharacterTextSplitter

#Ollama integartion
from langchain_ollama import OllamaEmbeddings, ChatOllama

#vectorestore
from langchain_community.vectorstores import Chroma
#from langchain_chroma import Chroma

#langchain core components
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

print("All import success")
import warnings
warnings.filterwarnings("ignore")


All import success


In [29]:
#Checks ollama running state and list of models
!ollama list

]11;?\NAME                       ID              SIZE      MODIFIED     
qwen3:1.7b                 8f68893c685c    1.4 GB    22 hours ago    
nomic-embed-text:latest    0a109f422b47    274 MB    23 hours ago    


In [30]:
#check a test query
print("Checking Ollama")
try:
    test_llm= ChatOllama(model= "qwen3:1.7b", temperature=0)
    response= test_llm.invoke("What is boiling point of water?")
    print(f"The response to your query:{response.content}")
except Exception as e:
    print(f"error loading the model{e}")

Checking Ollama
The response to your query:The **boiling point** of water is the temperature at which it **changes from a liquid to a gas** (vapor) under standard atmospheric pressure (1 atmosphere or 101.3 kPa). At this temperature, water boils and turns into steam. 

### Key Details:
1. **Standard Boiling Point**:  
   - At **1 atmosphere (101.3 kPa)**, water boils at **100°C (212°F)**.  
   - This is the **normal boiling point** of water.

2. **Effect of Pressure**:  
   - The boiling point **increases** with **higher pressure** (e.g., at sea level, 100°C; at higher altitudes, water boils at lower temperatures because atmospheric pressure is lower).  
   - Conversely, the boiling point **decreases** with **lower pressure** (e.g., in a vacuum, water boils at ~100°C, but at very low pressure, it boils at much lower temperatures).

3. **Practical Implications**:  
   - In cooking, boiling point varies with altitude. For example, at high altitudes (where pressure is lower), water boils 

In [31]:
#load pdf documents

pdf_path= "pdfs/rag.pdf"

if not os.path.exists(pdf_path):
    print("document not loaded")
else:
    print("document loaded success")
    #import pdf loader
    loader= PyPDFLoader(pdf_path)
    documents= loader.load()
    print(f"The loaded document has {len(documents)} pages from {pdf_path}")
    print(f"Content first(300 characters): {documents[0].page_content[:300]}")
    print(f"Metadata:{documents[0].metadata}")




document loaded success
The loaded document has 19 pages from pdfs/rag.pdf
Content first(300 characters): Retrieval-Augmented Generation for
Knowledge-Intensive NLP Tasks
Patrick Lewis†‡, Ethan Perez⋆,
Aleksandra Piktus†, Fabio Petroni†, Vladimir Karpukhin†, Naman Goyal†, Heinrich Küttler†,
Mike Lewis†, Wen-tau Yih†, Tim Rocktäschel†‡, Sebastian Riedel†‡, Douwe Kiela†
†Facebook AI Research;‡University C
Metadata:{'producer': 'pdfTeX-1.40.21', 'creator': 'LaTeX with hyperref', 'creationdate': '2021-04-13T00:48:38+00:00', 'author': '', 'keywords': '', 'moddate': '2021-04-13T00:48:38+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.14159265-2.6-1.40.21 (TeX Live 2020) kpathsea version 6.3.2', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'pdfs/rag.pdf', 'total_pages': 19, 'page': 0, 'page_label': '1'}


In [32]:
#Split document into chucnks

text_splitter= RecursiveCharacterTextSplitter(chunk_size= 1024,
                                              chunk_overlap= 128,
                                              length_function=len,
                                              separators=["\n\n","\n"," ",""])

#split the documents
chunks= text_splitter.split_documents(documents)

#display results
avg_chunk_size= sum(len(chunk.page_content) for chunk in chunks)/len(chunks)

print(f"avg_chunk is {avg_chunk_size:.0f}")

#Preview first three chunks

for i,chunk in enumerate(chunks[:3]):
    print(f"Chunk {i+1} (length: {len(chunk.page_content)})")
    print(f"{chunk.page_content[:200]}...")


avg_chunk is 877
Chunk 1 (length: 1006)
Retrieval-Augmented Generation for
Knowledge-Intensive NLP Tasks
Patrick Lewis†‡, Ethan Perez⋆,
Aleksandra Piktus†, Fabio Petroni†, Vladimir Karpukhin†, Naman Goyal†, Heinrich Küttler†,
Mike Lewis†, W...
Chunk 2 (length: 1023)
memory have so far been only investigated for extractive downstream tasks. We
explore a general-purpose ﬁne-tuning recipe for retrieval-augmented generation
(RAG) — models which combine pre-trained pa...
Chunk 3 (length: 988)
more speciﬁc, diverse and factual language than a state-of-the-art parametric-only
seq2seq baseline.
1 Introduction
Pre-trained neural language models have been shown to learn a substantial amount of ...


In [33]:
#Initialize Ollama embedding with nomic-embed-text
embeedding= OllamaEmbeddings(model= "nomic-embed-text:latest") 

#sample
test= "This is sample for testing the embedding"
sample_embed= embeedding.embed_query(test)

print(f"sample embedding dimension: {len(sample_embed)}")
print(f"First 10 values of sample embedding: {sample_embed[:10]}")

sample embedding dimension: 768
First 10 values of sample embedding: [0.019769674, 0.03997285, -0.17119965, -0.075656116, 0.043439813, -0.049264915, 0.027598573, -0.012623496, -0.018986998, -0.040743615]


In [39]:
#Create chromaDB vector store

print(f"creating chromaDB vectorstore using {len(chunks)} chunks")

#set persistent directory
persist_directory= "./chroma_db"

#create a vectorstore
vector_store= Chroma.from_documents(
    documents= chunks,
    embedding= embeedding,
    persist_directory=persist_directory,
    collection_name= "local_rag_collection"

)
print("chromadb vector store created successfully")
print(f"✓ ChromaDB vector store created successfully!")
print(f"✓ Indexed {len(chunks)} document chunks")
#print(f"✓ Stored at: {persist_directory}")

creating chromaDB vectorstore using 86 chunks
chromadb vector store created successfully
✓ ChromaDB vector store created successfully!
✓ Indexed 86 document chunks


In [35]:
#create retriever and test

retriever= vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k":4}) #retrieve top 4 relevant chunk

print("Retriever configured")

#test it
query= "What is open domain question answering?"
print(f"test query : {query}")
retrieved_doc= retriever.invoke(query)

print(f"test query answer: {retrieved_doc}")

print(f"\nRetrieved {len(retrieved_doc)} documents:")
for i, doc in enumerate(retrieved_doc):
    print(f"\nDocument {i+1}:")
    print(f"  Content preview: {doc.page_content[:150]}...")
    print(f"  Source: Page {doc.metadata.get('page', 'N/A')}")

Retriever configured
test query : What is open domain question answering?
test query answer: [Document(metadata={'producer': 'pdfTeX-1.40.21', 'trapped': '/False', 'title': '', 'creationdate': '2021-04-13T00:48:38+00:00', 'author': '', 'ptex.fullbanner': 'This is pdfTeX, Version 3.14159265-2.6-1.40.21 (TeX Live 2020) kpathsea version 6.3.2', 'subject': '', 'page': 3, 'creator': 'LaTeX with hyperref', 'page_label': '4', 'total_pages': 19, 'keywords': '', 'source': 'pdfs/rag.pdf', 'moddate': '2021-04-13T00:48:38+00:00'}, page_content='Open-domain question answering (QA) is an important real-world application and common testbed\nfor knowledge-intensive tasks [20]. We treat questions and answers as input-output text pairs (x,y )\nand train RAG by directly minimizing the negative log-likelihood of answers. We compare RAG to\nthe popular extractive QA paradigm [5, 7, 31, 26], where answers are extracted spans from retrieved\ndocuments, relying primarily on non-parametric knowledge. We also c

In [36]:
#Initiallize Ollama LLM

llm= ChatOllama(model="qwen3:1.7b",temperature=0)

print("LLM configured")

test_response= llm.invoke(query)
print(test_response)

LLM configured
content='**Open Domain Question Answering (ODQA)** refers to a type of question-answering system designed to handle queries across a broad spectrum of topics, rather than being confined to a specific domain. Here\'s a structured breakdown:\n\n### Key Characteristics:\n1. **Broad Training Data**:  \n   - Models are trained on large, diverse datasets (e.g., Wikipedia, news articles, books, and general knowledge sources) to ensure versatility. This prevents overfitting to a narrow domain.\n\n2. **General Knowledge Capabilities**:  \n   - Unlike closed-domain systems (e.g., medical or legal QA), open-domain models can answer questions about any topic, including complex or abstract subjects (e.g., "What is the history of the internet?").\n\n3. **Context and Reasoning**:  \n   - These systems often rely on contextual understanding, inference, and knowledge retrieval to answer questions. They may use techniques like pre-trained language models (e.g., BERT, GPT) or knowledge gra

In [37]:
#Build rag chain (Langchain Expression Language)

#define prompt template

system_prompt= ("you are a helpful assistant for question answering"
                "Use the following pieces of retrieved content to answer the question"
                "Keep the answer concise and accurate.\n\n"
                "Context: {context}\n\n"
                "Question:{question}"
                )
prompt= ChatPromptTemplate.from_template(system_prompt)

#Helper function to format documents
def format_docs(docs):
    """Format retrieved document into a single string"""
    return "\n\n" .join(doc.page_content for doc in docs)

#Build a rag chain using LCEL
rag_chain= (
    {
        "context": retriever|format_docs,
        "question": RunnablePassthrough()
    }
    |prompt
    |llm
    |StrOutputParser()
) 




In [38]:
#test rag pipeline

# Example Query 1: General question
query1 = "What is Open-domain Question Answering?"

print(f"Query: {query1}")
print("\nProcessing locally...\n")

answer = rag_chain.invoke(query1)

print("=" * 80)
print("ANSWER:")
print("=" * 80)
print(answer)
print("\n" + "=" * 80)

Query: What is Open-domain Question Answering?

Processing locally...

ANSWER:
Open-domain Question Answering (QA) is an application for knowledge-intensive tasks, where questions and answers are treated as input-output pairs. It uses retrieved documents to train models by minimizing answer likelihood, contrasting with extractive QA (extracting answers from texts) and closed-book QA (generating answers without retrieval). It's used as a testbed for evaluating NLP models' knowledge capabilities.

